# Model comparison

Everything here is scored on the 20k **validation** set. The test set is scored once, at the end.

## Day 3: baselines, then bag-of-words vs TF-IDF

Every text model uses the same cleaning (`reviews.text`), the same vocabulary (words seen at least twice, one-letter tokens kept) and the same logistic regression. Only the word weighting changes. Numbers come from `python -m reviews.compare`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from reviews.compare import load
from reviews.models import Vader
from reviews.scrub import excerpt

report = json.loads(Path("../reports/compare_bow_tfidf.json").read_text())
table = pd.DataFrame(report["results"]).T
table[["accuracy", "accuracy_95", "f1_negative", "f1_negative_95", "auc"]]

In [ ]:
report["paired_accuracy_difference_95"]

- **TF-IDF wins, by a little:** 90.4% against 89.7% for raw counts. The paired interval (same reviews resampled for both models) is +0.4 to +1.0 points, so the gap is real but small. Binary presence and counts can't be told apart.
- **The majority baseline's F1 of 0.67 is an artefact:** the train sample has 64 more negative than positive reviews (100,032 vs 99,968), so it calls everything negative. Recall 1, precision 0.5. That's why accuracy sits next to it.
- **Length alone: 53.5%.** As the EDA suggested, it's barely a signal here.
- **VADER, the word list, gets 72%**: well above chance, far below any trained model.

### Why VADER does badly on negative reviews

In [ ]:
val_x, val_y = load("validation")
vader_score = Vader().score(val_x)
vader_pred = (vader_score > 0).astype(int)
pd.crosstab(val_y, vader_pred, rownames=["true"], colnames=["VADER"])

In [ ]:
negative = val_y.to_numpy() == 0
share_very_positive = ((vader_score > 0.5) & negative).sum() / negative.sum()
round(share_very_positive, 3)

In [ ]:
# Picked on validation itself, so this flatters VADER: it's an upper bound, not a result.
thresholds = np.linspace(-0.5, 0.9, 29)
accuracy = [((vader_score > t).astype(int) == val_y).mean() for t in thresholds]
round(max(accuracy), 4), round(thresholds[int(np.argmax(accuracy))], 2)

In [ ]:
idx = np.where(negative & (vader_score > 0.8))[0][:4]
for i in idx:
    print(f"{vader_score[i]:+.2f} | {excerpt(val_x[i], 220)}")
    print()

Almost half of the negative reviews (49%) come out positive, and a third score above +0.5. Negative product reviews are full of positive words, as the examples show: what other reviewers said ("The reviews looked great"), praise before the "but" ("Cute film... unfortunately"), or a story with no verdict at all. A word list adds the words up; a trained model learns that "disappointing" and "unfortunately" outweigh them. Some, like the second example, read as label noise rather than a VADER mistake. Even with the best threshold picked on validation itself, VADER tops out at 77%.